[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_demo09.ipynb)

# MATH840 — Week 9 practice: Prophet up close, then the wisdom of averages

**Not marked.** Two parts, one question running through both: *does the new tool actually beat the old
ones?*

**Part 1 — Prophet, piece by piece.** Facebook's forecasting library promises a lot: several seasons at
once, holidays, outside variables, a trend that bends. We switch those features on one at a time and,
after each one, check the score against the models you already know.

**Part 2 — combining.** Instead of crowning one model, average several. Why that works, when it does not,
and the recipe you will use in tonight's Lab 8.

| | Story | What you will see | Min |
|---|---|---|---:|
| 1 | ⚡ Melbourne's electricity, day by day | Prophet's parts: seasons, holidays, temperature | 30 |
| 2 | 🍽️ Australians eating out | Prophet's trend and the two settings that make or break it | 15 |
| 3 | 🍽️ the same series, four models | why an average of forecasts is hard to beat | 10 |
| 4 | 🏨 76 tourist regions | choose the best model, or average them all? | 10 |
| 5 | 🍽️ the Lab 8 recipe | combination + an honest interval, step by step | 20 |

🔮 marks a guess: write it down before running the cell. ❓ marks a question for the room.

## 0. Setup

In [ ]:
!pip install -q statsforecast==2.1.1 great_tables prophet

import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import logging
import warnings
warnings.filterwarnings("ignore")
for name in ("cmdstanpy", "prophet"):
    logging.getLogger(name).setLevel(logging.CRITICAL)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from great_tables import GT, loc, style
from matplotlib.colors import LinearSegmentedColormap, to_hex
from prophet import Prophet
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, MSTL, SeasonalNaive

plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 120, "display.precision", 3)

BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")
COLOURS = {"actual": "#314f4f", "a": "#e64173", "b": "#20B2AA", "c": "#FFA500", "d": "#6A5ACD", "old": "#9aa5a5"}

Three helpers for the whole session. `backtest` is the important one: it takes **any** forecaster — a
function that gets "how much history you may see" and returns the next `h` values — so ETS and Prophet
are scored by exactly the same rule.

In [ ]:
def backtest(y, forecasters, origins, h, scale, labels=None):
    # MASE of every forecaster from every origin, plus the mean; the forecasts come back for plotting.
    forecasts = {name: [np.asarray(f(e)) for e in origins] for name, f in forecasters.items()}
    table = pd.DataFrame({name: [np.mean(np.abs(y[e:e + h] - fc)) / scale for e, fc in zip(origins, fcs)]
                          for name, fcs in forecasts.items()}, index=labels or origins).T
    table["mean"] = table.mean(axis=1)
    return table, forecasts


def fourier(t, period, K):
    # Fourier terms from Week 5; the period can be any number: 7, 365.25, 52.18.
    return np.column_stack([f(2 * np.pi * k * t / period) for k in range(1, K + 1) for f in (np.sin, np.cos)])


_shade = LinearSegmentedColormap.from_list("shade", ["#bfe0b0", "#ffffff", "#f5b8cb"])


def scoreboard(table, rules=None, decimals=2):
    # Green is better, pink is worse, the best in bold. A rule is "lower", "higher" or a target value.
    df = table.to_frame() if isinstance(table, pd.Series) else table.copy()
    df.index, df.columns = df.index.map(str), df.columns.map(str)
    rules = rules or {c: "lower" for c in df.columns}
    gt = (GT(df.reset_index(names=" "), rowname_col=" ")
          .fmt_number(columns=list(df.select_dtypes("number").columns), decimals=decimals)
          .sub_missing(missing_text=""))
    for c, rule in rules.items():
        v = df[c].astype(float)
        bad = v if rule == "lower" else -v if rule == "higher" else (v - rule).abs()
        lo, hi = bad.min(), bad.max()
        for i, b in enumerate(bad):
            if np.isnan(b):
                continue
            look = [style.fill(color=to_hex(_shade((b - lo) / (hi - lo) if hi > lo else 0.0)))]
            gt = gt.tab_style(style=look + ([style.text(weight="bold")] if b == lo else []),
                              locations=loc.body(columns=c, rows=[i]))
    return gt

# Part 1. Prophet, piece by piece

## 1. ⚡ Melbourne's electricity, day by day

You run the power grid of the state of Victoria. Every morning someone asks you: *how much electricity
will we need each day for the next four weeks?* Too little, and you buy at panic prices or cut people off;
too much, and you pay for power plants standing idle.

The data: daily demand in 2012–2014 (GWh), the day's maximum temperature, and the public-holiday calendar.

In [ ]:
vic = pd.read_csv(f"{BASE}/vic_elec.csv", parse_dates=["ds"])
wide = vic.pivot_table(index="ds", columns="unique_id", values="y")
holiday = vic.query("unique_id == 'Demand'").set_index("ds")["Holiday"].astype(bool)
elec = pd.DataFrame({"y": wide["Demand"].resample("D").sum() / 2e3,      # half-hourly MW -> GWh per day
                     "temp": wide["Temperature"].resample("D").max(),
                     "holiday": holiday.resample("D").max()}).reset_index()
elec["temp2"] = elec["temp"] ** 2

fig, axes = plt.subplots(1, 2, figsize=(14, 3.4), width_ratios=[2.3, 1])
axes[0].plot(elec["ds"], elec["y"], color=COLOURS["actual"], linewidth=0.7)
axes[0].set_title("Daily demand, GWh: a weekly zigzag riding on a yearly wave")
axes[1].scatter(elec["temp"], elec["y"], s=5, color=COLOURS["b"], alpha=0.5)
axes[1].set_title("Demand against the day's temperature")
axes[1].set_xlabel("maximum temperature, °C")
plt.show()

Two seasons at once — **weekly** (weekends are quiet) and **yearly** (winter heating, summer
air-conditioning) — and a **U-shaped** link with temperature: both cold and hot days push demand up.

ETS and seasonal ARIMA hold **one** season. Prophet was built for exactly this kind of series.

### 1.1 Prophet in five lines

Prophet does not look at yesterday's value the way ETS and ARIMA do. It draws a curve through time:

$$y_t = \underbrace{g(t)}_{\text{trend}} + \underbrace{s(t)}_{\text{seasons}} + \underbrace{h(t)}_{\text{holidays}}
+ \underbrace{\beta x_t}_{\text{extra variables}} + \varepsilon_t$$

Each season is a sum of sines and cosines — the Fourier terms from Week 5. Let's fit it on everything up to
the end of September 2014 and forecast October–December.

In [ ]:
cut = int(elec["ds"].searchsorted(pd.Timestamp("2014-10-01")))
train_e, test_e = elec.iloc[:cut], elec.iloc[cut:]

m = Prophet(daily_seasonality=False)            # weekly and yearly seasons switch on by themselves
m.fit(train_e[["ds", "y"]])
fc = m.predict(test_e[["ds"]])

inside = np.mean((test_e["y"].to_numpy() >= fc["yhat_lower"]) & (test_e["y"].to_numpy() <= fc["yhat_upper"]))
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(elec["ds"].iloc[cut - 90:], elec["y"].iloc[cut - 90:], color=COLOURS["actual"], linewidth=1, label="actual")
ax.plot(fc["ds"], fc["yhat"], color=COLOURS["a"], linewidth=1.6, label="Prophet")
ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"], color=COLOURS["a"], alpha=0.15, label="80% interval")
ax.axvline(test_e["ds"].iloc[0], color="grey", linewidth=0.8)
ax.set_title(f"Default Prophet, October to December 2014: {inside:.0%} of days inside the 80% interval")
ax.legend(loc="upper left", fontsize=8)
plt.show()

### 1.2 Open the box: what did Prophet learn?

The big advantage of Prophet over ARIMA: every piece of the curve can be drawn and read.

🔮 **Guess first.** Which day of the week is the quietest? In which month does Melbourne use the most
electricity — the height of summer (January) or the depth of winter (July)?

In [ ]:
fig = m.plot_components(m.predict(elec[["ds"]]), figsize=(11, 7))      # drawn over 2012-2014
plt.show()

### 1.3 Holidays: the days the calendar breaks

On a public holiday the city behaves like on a Sunday — but the weekly season thinks it is a Thursday.
How big is that mistake? Look at the default model's errors on the training days.

In [ ]:
fit_e = m.predict(train_e[["ds"]])
residual = train_e["y"].to_numpy() - fit_e["yhat"].to_numpy()
on_holiday = train_e["holiday"].to_numpy()
print(f"ordinary days: on average the model is off by {residual[~on_holiday].mean():+.1f} GWh")
print(f"public holidays: on average the model overshoots by {-residual[on_holiday].mean():.1f} GWh")

Prophet takes holidays as a table: a name and a date, and optionally a **window** of days around it. We
give each Victorian holiday its name — and to Christmas a window of six days after it, because the whole
week between Christmas and New Year is quiet.

In [ ]:
def holiday_name(d):
    if d.month == 1:
        return "New Year" if d.day <= 2 else "Australia Day"
    if d.month == 3 and d.day <= 14:
        return "Labour Day"
    if (d.month, d.day) == (4, 25):
        return "Anzac Day"
    if d.month in (3, 4):
        return "Easter"
    return {6: "Queen's Birthday", 11: "Melbourne Cup"}.get(d.month, "Christmas" if d.day == 25 else "Boxing Day")


days_off = elec.loc[elec["holiday"], "ds"]
holidays = pd.DataFrame({"holiday": [holiday_name(d) for d in days_off], "ds": days_off.to_numpy(),
                         "lower_window": 0, "upper_window": 0})
holidays = holidays[holidays["holiday"] != "Boxing Day"]                # Christmas's window covers it
holidays.loc[holidays["holiday"] == "Christmas", "upper_window"] = 6

m_hol = Prophet(daily_seasonality=False, holidays=holidays).fit(train_e[["ds", "y"]])
parts = m_hol.predict(train_e[["ds"]])
effect = pd.Series({name: parts.loc[train_e["ds"].isin(holidays.loc[holidays["holiday"] == name, "ds"]).to_numpy(), name].mean()
                    for name in holidays["holiday"].unique()}).sort_values()
effect.plot.barh(color=COLOURS["b"], figsize=(8, 3.2), title="What each holiday does to demand, GWh")
plt.show()

### 1.4 Temperature: an outside variable

The U-shape from the first plot cannot come from the calendar. Prophet takes extra columns with
`add_regressor`. A straight line cannot bend into a U, so we give it the temperature **and** its square.

🔮 **Guess first.** At what temperature is demand lowest?

In [ ]:
m_temp = Prophet(daily_seasonality=False, holidays=holidays)
m_temp.add_regressor("temp")
m_temp.add_regressor("temp2")
m_temp.fit(train_e[["ds", "y", "temp", "temp2"]])
parts = m_temp.predict(train_e[["ds", "temp", "temp2"]])

order = np.argsort(train_e["temp"].to_numpy())
curve = parts["extra_regressors_additive"].to_numpy()[order]
lowest = train_e["temp"].to_numpy()[order][np.argmin(curve)]
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(train_e["temp"].to_numpy()[order], curve, color=COLOURS["a"], linewidth=2)
ax.axvline(lowest, color="grey", linestyle=":")
ax.set_xlabel("maximum temperature, °C")
ax.set_ylabel("GWh")
ax.set_title(f"What temperature adds to demand: lowest at about {lowest:.0f} °C")
plt.show()

### 1.5 The scoreboard: Prophet against the old guard

Nine four-week windows from April to December 2014. On one side, models you know:

- **seasonal naive** — the same day last week;
- **ETS** and **ARIMA** with $m = 7$ — one season only;
- **MSTL** with both seasons, 7 and 365;
- **harmonic regression** — Fourier terms for both seasons, with ARIMA errors (Week 5): Prophet's idea,
  made by hand;
- **harmonic + weather** — the same, plus temperature, its square and the holiday flag.

On the other, Prophet with one feature more at each step — and one honest version. The weather models use
the temperature that **actually happened** — an ex-post forecast (Week 7). On 4 December nobody knows
what the temperature will be on the 30th. So the last Prophet uses **the climate instead**: the average
temperature of the same week in earlier years.

In [ ]:
y1 = elec["y"].to_numpy()
t1 = np.arange(len(y1))
H1 = 28
origins1 = [len(y1) - H1 * k for k in range(9, 0, -1)]
labels1 = [f"{elec['ds'][e]:%d %b}" for e in origins1]
scale1 = np.mean(np.abs(y1[7:origins1[0]] - y1[:origins1[0] - 7]))      # seasonal naive MAE before the tests

X_season = np.column_stack([fourier(t1, 7, 3), fourier(t1, 365.25, 6)])
X_weather = np.column_stack([X_season, elec["temp"], elec["temp2"], elec["holiday"]])


def harmonic(X):
    return lambda e: AutoARIMA(season_length=1, d=0, max_p=3, max_q=3).fit(
        y=y1[:e], X=X[:e]).predict(h=H1, X=X[e:e + H1])["mean"]


def climate(e):
    # The forecaster's temperature on day e: the mean of the same calendar week in the years before.
    past = elec.iloc[:e]
    doy = past["ds"].dt.dayofyear.to_numpy()
    out = []
    for day in elec["ds"].iloc[e:e + H1].dt.dayofyear:
        gap = np.abs(doy - day)
        out.append(past["temp"].to_numpy()[np.minimum(gap, 365 - gap) <= 3].mean())
    return np.array(out)


def prophet_elec(with_holidays=False, weather=None):
    def forecast(e):
        model = Prophet(daily_seasonality=False, holidays=holidays if with_holidays else None)
        future = elec.iloc[e:e + H1].copy()
        if weather:
            model.add_regressor("temp")
            model.add_regressor("temp2")
            if weather == "climate":
                future["temp"] = climate(e)
                future["temp2"] = future["temp"] ** 2
        model.fit(elec.iloc[:e])
        return model.predict(future)["yhat"].to_numpy()
    return forecast


forecasters1 = {
    "seasonal naive": lambda e: np.tile(y1[e - 7:e], 4),
    "ETS (m = 7)": lambda e: AutoETS(season_length=7).fit(y=y1[:e]).predict(h=H1)["mean"],
    "ARIMA (m = 7)": lambda e: AutoARIMA(season_length=7).fit(y=y1[:e]).predict(h=H1)["mean"],
    "MSTL (7, 365)": lambda e: MSTL(season_length=[7, 365]).fit(y=y1[:e]).predict(h=H1)["mean"],
    "harmonic regression": harmonic(X_season),
    "harmonic + weather (actual)": harmonic(X_weather),
    "Prophet": prophet_elec(),
    "Prophet + holidays": prophet_elec(True),
    "Prophet + holidays + weather (actual)": prophet_elec(True, "actual"),
    "Prophet + holidays + weather (climate)": prophet_elec(True, "climate"),
}

🔮 **Guess first.** Ten forecasters, nine windows. Put them in order. Does Prophet beat MSTL? Does it beat
the harmonic regression, which is built from the same parts? How much of the weather's gain survives
when the weather is not known in advance? (About two minutes.)

In [ ]:
scores1, fc1 = backtest(y1, forecasters1, origins1, H1, scale1, labels1)
scoreboard(scores1)

In [ ]:
means = scores1["mean"].iloc[::-1]
colours = [COLOURS["a"] if n.startswith("Prophet") else COLOURS["old"] for n in means.index]
fig, ax = plt.subplots(figsize=(10, 4.2))
bars = ax.barh(means.index, means, color=colours)
for bar, name in zip(bars, means.index):
    if "(actual)" in name:
        bar.set_hatch("//")
        bar.set_edgecolor("white")
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
ax.set_xlabel("mean MASE over nine windows (shorter is better)")
ax.set_title("Pink: Prophet; grey: the models you knew; hatched: uses the real future temperature")
plt.show()

**❓ Question.** You would hand the grid operator Prophet with the weather. Which temperature do you feed
it on 4 December — and what would you show them instead of one line?

## 2. 🍽️ Australians eating out: Prophet's trend

Monthly turnover of all cafés, restaurants and takeaways in Australia, 1982–2018, $ million. A completely
different series: one season, but a long trend that bends — a recession in the early 1990s, a GST in
2000, the 2008 crisis.

Two settings decide whether Prophet works here:

- `seasonality_mode` — does the December peak add a fixed amount (**additive**, the default) or a
  fixed share (**multiplicative**)?
- `changepoint_prior_scale` — how freely the trend may change slope (default 0.05).

In [ ]:
retail = pd.read_csv(f"{BASE}/aus_retail.csv", parse_dates=["Month"])
national = (retail.groupby(["Industry", "Month"], as_index=False)["Turnover"].sum()
            .rename(columns={"Industry": "unique_id", "Month": "ds", "Turnover": "y"}))
cafes = national.query("unique_id == 'Cafes, restaurants and takeaway food services'")[["ds", "y"]].reset_index(drop=True)
train_c, test_c = cafes.iloc[:-12], cafes.iloc[-12:]                  # 2018 is the test year


def prophet_cafes(**settings):
    return Prophet(weekly_seasonality=False, daily_seasonality=False, **settings)


fits = {mode: prophet_cafes(seasonality_mode=mode).fit(train_c) for mode in ("additive", "multiplicative")}
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
for ax, (start, end), title in [(axes[0], ("1982-04", "1989-12"), "The 1980s: in-sample fit"),
                                (axes[1], ("2014-01", "2018-12"), "2018: the forecast")]:
    span = cafes[(cafes["ds"] >= start) & (cafes["ds"] <= end)]
    ax.plot(span["ds"], span["y"], color=COLOURS["actual"], linewidth=1.5, label="actual")
    for mode, colour in [("additive", COLOURS["c"]), ("multiplicative", COLOURS["b"])]:
        p = fits[mode].predict(span[["ds"]])
        ax.plot(p["ds"], p["yhat"], color=colour, linewidth=1.3, label=mode)
    ax.set_title(title)
axes[1].axvline(test_c["ds"].iloc[0], color="grey", linewidth=0.8)
axes[0].legend(fontsize=8)
plt.show()

The additive season is a compromise between a small December in 1985 and a big one in 2017: too big at
the start, too small at the end. The multiplicative one scales with the level, the way a log does in ETS
and ARIMA.

Now the trend. 🔮 **Guess first.** With the trend almost frozen (0.001) or very loose (0.5), what does
Prophet predict for the next five years?

In [ ]:
ahead = pd.DataFrame({"ds": pd.date_range(cafes["ds"].iloc[0], periods=len(train_c) + 60, freq="MS")})
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cafes["ds"], cafes["y"], color=COLOURS["actual"], linewidth=0.8, alpha=0.6, label="actual")
for scale, colour in [(0.001, COLOURS["c"]), (0.05, COLOURS["b"]), (0.5, COLOURS["a"])]:
    model = prophet_cafes(seasonality_mode="multiplicative", changepoint_prior_scale=scale).fit(train_c)
    ax.plot(ahead["ds"], model.predict(ahead)["trend"], color=colour, linewidth=2,
            label=f"trend, changepoint_prior_scale = {scale}")
    if scale == 0.05:
        bends = model.changepoints[np.abs(model.params["delta"]).mean(axis=0) > 0.01]
        for d in bends:
            ax.axvline(d, color=COLOURS["b"], linestyle=":", linewidth=0.8)
ax.axvline(test_c["ds"].iloc[0], color="grey", linewidth=0.8)
ax.set_title("Three trends from the same data (dotted: where the default trend bends)")
ax.legend(loc="upper left", fontsize=8)
plt.show()

Six rolling 12-month windows, 2013–2018, against the models you fitted in Labs 5 and 7. The log
versions of ARIMA and MSTL are there because that is what a careful analyst would use on this series.

In [ ]:
y2 = cafes["y"].to_numpy()
H2 = 12
origins2 = [len(y2) - H2 * k for k in range(6, 0, -1)]
labels2 = [f"{cafes['ds'][e]:%Y}" for e in origins2]
scale2 = np.mean(np.abs(y2[12:origins2[0]] - y2[:origins2[0] - 12]))


def prophet_rolling(**settings):
    return lambda e: prophet_cafes(**settings).fit(cafes.iloc[:e]).predict(cafes.iloc[e:e + H2][["ds"]])["yhat"].to_numpy()


def logged(model):
    return lambda e: np.exp(model().fit(y=np.log(y2[:e])).predict(h=H2)["mean"])


forecasters2 = {
    "seasonal naive": lambda e: y2[e - 12:e],
    "ETS": lambda e: AutoETS(season_length=12).fit(y=y2[:e]).predict(h=H2)["mean"],
    "ARIMA (log)": logged(lambda: AutoARIMA(season_length=12)),
    "MSTL (log)": logged(lambda: MSTL(season_length=12)),
    "Prophet, defaults": prophet_rolling(),
    "Prophet, multiplicative": prophet_rolling(seasonality_mode="multiplicative"),
    "Prophet, multiplicative, loose trend (0.5)": prophet_rolling(seasonality_mode="multiplicative", changepoint_prior_scale=0.5),
    "Prophet, multiplicative, frozen trend (0.001)": prophet_rolling(seasonality_mode="multiplicative", changepoint_prior_scale=0.001),
}
scores2, fc2 = backtest(y2, forecasters2, origins2, H2, scale2, labels2)
scoreboard(scores2)

### Prophet in one table

| Prophet is good at | Prophet is weak at |
|---|---|
| several seasons at once, of any length (7, 365.25, 52.18) | it never looks at the last value: no memory of a shock yesterday |
| holidays, with windows around them | the trend is extrapolated in a straight line, and its flexibility is a guess |
| outside variables (temperature, prices, promotions) | the defaults: additive season, one fixed `changepoint_prior_scale` |
| missing days and irregular dates | its interval covers only noise and trend changes, not wrong seasons |
| every piece can be drawn and explained to a manager | on a plain monthly series, a well-chosen ETS or ARIMA is usually as good or better |

**❓ Question.** The harmonic regression with weather scored about the same as Prophet with weather. So what
did Prophet give you that the regression did not?

# Part 2. Combining forecasts

## 3. 🍽️ Four forecasts of 2018 — and their average

In part 1 every model was a contestant and we picked a winner. Now a different idea, as old as
forecasting itself (Bates and Granger, 1969): **take several decent forecasts and average them.**

Four models, all fitted on data up to December 2017: ETS, ARIMA and MSTL on the log, and the
multiplicative Prophet from case 2. The test is 2018.

🔮 **Guess first.** Will the average of the four beat the best of them? The worst? Where will it land?

In [ ]:
past_c = train_c["y"].to_numpy()
actual_c = test_c["y"].to_numpy()
scale_c = np.mean(np.abs(past_c[12:] - past_c[:-12]))


def on_log(model, history, h=12):
    return np.exp(model.fit(y=np.log(history)).predict(h=h)["mean"])


four = {
    "ETS": on_log(AutoETS(season_length=12), past_c),
    "ARIMA": on_log(AutoARIMA(season_length=12), past_c),
    "MSTL": on_log(MSTL(season_length=12), past_c),
    "Prophet": prophet_cafes(seasonality_mode="multiplicative").fit(train_c).predict(test_c[["ds"]])["yhat"].to_numpy(),
}
four["average"] = np.mean([four[n] for n in ("ETS", "ARIMA", "MSTL", "Prophet")], axis=0)
mase_c = pd.Series({n: np.mean(np.abs(actual_c - f)) / scale_c for n, f in four.items()})

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), width_ratios=[1.6, 1])
axes[0].plot(test_c["ds"], actual_c, color=COLOURS["actual"], linewidth=2.5, label="actual")
for (name, f), colour in zip(four.items(), [COLOURS["d"], COLOURS["b"], COLOURS["c"], COLOURS["old"], COLOURS["a"]]):
    axes[0].plot(test_c["ds"], f, color=colour, linewidth=2.5 if name.startswith("average") else 1.2,
                 linestyle="-" if name.startswith("average") else "--", label=name)
axes[0].set_title("2018, $ million a month")
axes[0].legend(fontsize=8, ncol=2)
members_mean = mase_c.drop("average").mean()
bars = axes[1].bar(mase_c.index, mase_c, color=[COLOURS["old"]] * 4 + [COLOURS["a"]])
axes[1].bar_label(bars, fmt="%.2f", fontsize=9)
axes[1].axhline(members_mean, color="black", linestyle=":", linewidth=1)
axes[1].text(-0.4, members_mean * 1.03, f"the average member: {members_mean:.2f}", fontsize=9)
axes[1].set_title("MASE in 2018")
axes[1].tick_params(axis="x", labelrotation=20)
plt.show()

Why does it work? Each model is wrong in its own way: one overshoots December, another undershoots March.
In the average, errors of opposite sign **cancel**. And one guarantee holds always, for any series:

$$\Big|\,y - \tfrac{1}{k}\textstyle\sum_i \hat y_i\Big| \;\le\; \tfrac{1}{k}\textstyle\sum_i |y - \hat y_i|,$$

so the combination's MAE is **never worse than the average member's** (the dotted line). It can be
worse than the best member — but you do not know in advance who the best member will be.

## 4. 🏨 76 tourist regions: choose, or average?

"But I *can* find out who is best," you say. "I'll check on validation windows and take the winner."
Let's test exactly that — not on one series, where luck decides, but on 76.

Quarterly overnight trips to each of Australia's 76 tourism regions, 1998–2017. For every region:

- **choose** — score ETS, ARIMA and Theta on three validation windows and take the winner to the test;
- **average** — average the three, choose nothing.

The test is the last two years (8 quarters), never seen by either.

In [ ]:
tourism = pd.read_csv(f"{BASE}/tourism.csv", parse_dates=["ds"])
regions = tourism.groupby(["Region", "ds"], as_index=False)["y"].sum().rename(columns={"Region": "unique_id"})
H4 = 8
test4 = regions.groupby("unique_id").tail(H4)
train4 = regions.drop(test4.index)
names4 = ["ETS", "ARIMA", "Theta"]
sf4 = StatsForecast(models=[AutoETS(season_length=4, alias="ETS"), AutoARIMA(season_length=4, alias="ARIMA"),
                            AutoTheta(season_length=4, alias="Theta"), SeasonalNaive(4, alias="seasonal naive")],
                    freq="QS", n_jobs=-1)
validation = sf4.cross_validation(df=train4, h=H4, n_windows=3, step_size=4)       # three origins, a year apart
tested = sf4.forecast(df=train4, h=H4).merge(test4, on=["unique_id", "ds"])
scale4 = train4.groupby("unique_id")["y"].apply(lambda s: np.mean(np.abs(s.to_numpy()[4:] - s.to_numpy()[:-4])))


def mase_by_region(df, columns):
    return df.groupby("unique_id").apply(
        lambda g: pd.Series({c: np.mean(np.abs(g["y"] - g[c])) for c in columns}), include_groups=False
    ).div(scale4, axis=0)


tested["average of three"] = tested[names4].mean(axis=1)
val4 = mase_by_region(validation, names4)
test_scores4 = mase_by_region(tested, names4 + ["seasonal naive", "average of three"])
picked = val4.idxmin(axis=1)
test_scores4["choose on validation"] = [test_scores4.loc[r, picked[r]] for r in test_scores4.index]
test_scores4["oracle: knows the test winner"] = test_scores4[names4].min(axis=1)

🔮 **Guess first.** With three candidates, a coin picks the test winner one time in three. How often does
the validation pick it?

In [ ]:
hit = (picked == test_scores4[names4].idxmin(axis=1)).mean()
avg_wins = (test_scores4["average of three"] < test_scores4["choose on validation"]).mean()

means4 = test_scores4.mean().sort_values(ascending=False)
colours = [COLOURS["a"] if n == "average of three" else COLOURS["d"] if n.startswith("choose")
           else COLOURS["b"] if n.startswith("oracle") else COLOURS["old"] for n in means4.index]
fig, ax = plt.subplots(figsize=(10, 3.8))
bars = ax.barh(means4.index, means4, color=colours)
ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
ax.set_xlim(0.8, means4.max() * 1.08)
ax.set_xlabel("mean test MASE over 76 regions (shorter is better)")
ax.set_title(f"Validation picked the test winner in {hit:.0%} of regions (a coin: 33%); "
             f"the average beat the choice in {avg_wins:.0%}")
plt.show()

**❓ Question.** Theta on its own did better than both strategies. Should you just always use Theta? What
would you have needed to know in 2015 to make that call?

## 5. 🍽️ The Lab 8 recipe, step by step

Back to the cafés, and exactly what you will do tonight on your own series:

1. **Members.** ETS, ARIMA and MSTL — decided **before** seeing any scores.
2. **Rolling windows.** Every member forecasts four 12-month windows before the test (2014–2017).
3. **Read the table.** Who wins which window? How close is the combination to the best?
4. **The interval.** Two recipes, each judged on a window it has not seen.
5. **Hand in.** Refit on everything, average, add the interval — and see how 2018 went.

The helper functions below are the same as in the Lab 8 template.

In [ ]:
MEMBERS = {"ETS": lambda: AutoETS(season_length=12), "ARIMA": lambda: AutoARIMA(season_length=12),
           "MSTL": lambda: MSTL(season_length=12)}                      # all three on the log scale
WINDOWS, H5 = 4, 12


def member_forecast(make, history, h=H5):
    out = make().fit(y=np.log(history)).predict(h=h, level=[80])
    return {k: np.exp(v) for k, v in out.items()}


def backtest_members(history, windows=WINDOWS, h=H5):
    # One row per (window, step): the actual value and every member's forecast and 80% bounds.
    frames = []
    for k in range(windows, 0, -1):
        end = len(history) - (k - 1) * h
        past, actual = history[:end - h], history[end - h:end]
        frame = pd.DataFrame({"window": k, "step": np.arange(1, h + 1), "actual": actual})
        for name, make in MEMBERS.items():
            fc = member_forecast(make, past, h)
            frame[name], frame[f"{name} lo"], frame[f"{name} hi"] = fc["mean"], fc["lo-80"], fc["hi-80"]
        frames.append(frame)
    bt = pd.concat(frames, ignore_index=True)
    bt["combination"] = bt[list(MEMBERS)].mean(axis=1)
    bt["combination lo"] = bt[[f"{n} lo" for n in MEMBERS]].mean(axis=1)
    bt["combination hi"] = bt[[f"{n} hi" for n in MEMBERS]].mean(axis=1)
    return bt


def winkler(actual, lo, hi, alpha=0.2):
    # Width plus 2/alpha times every miss, in units of the MASE scale.
    return float(np.mean((hi - lo) + 2 / alpha * (np.maximum(lo - actual, 0) + np.maximum(actual - hi, 0))) / scale_c)


def conformal_radius(errors):
    # 80% of the past absolute errors were smaller than this: forecast +- radius is the 80% interval.
    return float(np.quantile(np.abs(errors), 0.80))


bt = backtest_members(past_c)
columns = list(MEMBERS) + ["combination"]
table5 = bt.groupby("window").apply(
    lambda g: pd.Series({c: np.mean(np.abs(g["actual"] - g[c])) / scale_c for c in columns}), include_groups=False).T
table5 = table5[sorted(table5.columns, reverse=True)]                    # oldest window first
table5.columns = [f"window {w}: {2018 - w}" for w in table5.columns]
table5["mean"] = table5.mean(axis=1)

🔮 **Guess first.** Four windows, three members. Will one member win all four? Where will the combination
be in each?

In [ ]:
scoreboard(table5)

In [ ]:
errors = bt[list(MEMBERS)].rsub(bt["actual"], axis=0)
print("how alike are the members' errors (correlation):")
print(errors.corr().round(2))

**The interval.** The members' own intervals know nothing about each other, and each believes its model is
right. Two recipes:

- **averaged bounds** — the mean of the members' lower bounds and of their upper bounds;
- **conformal** — take the combination's errors on the windows, find the size that 80% of them did not
  exceed, and put that radius on both sides of the new forecast. No model assumptions: only "the future
  errors will be like the past ones".

To judge fairly, each window is left out in turn: the conformal radius for window 1 comes from windows 2–4
only. Otherwise the recipe would be marking its own homework.

In [ ]:
rows = []
for w, g in bt.groupby("window"):
    others = bt[bt["window"] != w]
    r = conformal_radius(others["actual"] - others["combination"])
    for recipe, lo, hi in [("averaged bounds", g["combination lo"], g["combination hi"]),
                           ("conformal", g["combination"] - r, g["combination"] + r)]:
        rows.append({"recipe": recipe, "coverage": np.mean((g["actual"] >= lo) & (g["actual"] <= hi)),
                     "Winkler": winkler(g["actual"], lo, hi), "width": np.mean(hi - lo) / scale_c})
interval_check = pd.DataFrame(rows).groupby("recipe").mean()
scoreboard(interval_check, rules={"coverage": 0.8, "Winkler": "lower", "width": "lower"})

🔮 **Guess first.** Now the real thing: refit on everything up to December 2017, forecast 2018. Will
the combination beat the member that won the validation table? Which interval will hold 2018?

In [ ]:
final = {name: member_forecast(make, past_c) for name, make in MEMBERS.items()}
combo = np.mean([final[n]["mean"] for n in MEMBERS], axis=0)
radius = conformal_radius(bt["actual"] - bt["combination"])               # now from all four windows
intervals = {"averaged bounds": (np.mean([final[n]["lo-80"] for n in MEMBERS], axis=0),
                                 np.mean([final[n]["hi-80"] for n in MEMBERS], axis=0)),
             "conformal": (combo - radius, combo + radius)}
winner = table5.loc[list(MEMBERS), "mean"].idxmin()

test_table = pd.DataFrame({
    "MASE": {**{n: np.mean(np.abs(actual_c - final[n]["mean"])) / scale_c for n in MEMBERS},
             "combination": np.mean(np.abs(actual_c - combo)) / scale_c}})
test_table.loc[f"choose on validation ({winner})"] = test_table.loc[winner, "MASE"]
display(scoreboard(test_table))
scoreboard(pd.DataFrame({k: {"coverage": np.mean((actual_c >= lo) & (actual_c <= hi)),
                             "Winkler": winkler(actual_c, lo, hi), "width": np.mean(hi - lo) / scale_c}
                         for k, (lo, hi) in intervals.items()}).T,
           rules={"coverage": 0.8, "Winkler": "lower", "width": "lower"})

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8))
recent = cafes.iloc[-48:]
ax.plot(recent["ds"], recent["y"], color=COLOURS["actual"], linewidth=1.4, label="actual")
ax.plot(test_c["ds"], combo, color=COLOURS["a"], linewidth=2, label="combination")
for (name, (lo, hi)), colour, alpha in zip(intervals.items(), [COLOURS["b"], COLOURS["c"]], [0.2, 0.35]):
    ax.fill_between(test_c["ds"], lo, hi, color=colour, alpha=alpha, label=name)
ax.axvline(test_c["ds"].iloc[0], color="grey", linewidth=0.8)
ax.set_title("2018: the combination and its two 80% intervals")
ax.legend(loc="upper left", fontsize=8)
plt.show()

**❓ Question.** Your Lab 8 series is not the cafés. Which of tonight's decisions do you make **before**
seeing your table — which members, log or not, combine at all — and which does the table decide?

## What to take into tonight's lab

- **Prophet** is a curve-fitter with good parts: several seasons, holidays, outside variables. Its
  defaults can wreck a monthly series; switch on what the series needs and check it on rolling windows,
  like any other model.
- **Averaging** a few decent models is never worse than the average member, and on most series it lands
  close to the best one — without having to guess which one that is.
- **Choosing on validation** is noisier than it feels: three windows tell you less than you think.
- **An interval** should be judged on data it has not seen. Conformal is simple and honest about its
  own track record.

**Lab 8** is case 5 on your own Track B series — with your own ETS from Lab 5 and ARIMA from Lab 7.